# imports

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import (
    classification_report, accuracy_score,
    precision_score, recall_score, f1_score,
    confusion_matrix
)
import mlflow
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

c:\Users\Manindra\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# config

In [2]:
ASSET = "BTC"
INTERVAL = "4h"
EXPERIMENT = "confidence_threshold"
SOURCE_MODEL = "v4_native"

BEST_PARAMS = {
    'n_estimators': 100,
    'learning_rate': 0.01,
    'max_depth': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
}

BASELINE_V4_ACCURACY = 0.5274   # v4 native 4h tuned accuracy
BASELINE_V3_ACCURACY = 0.5243   # v3 derived 4h

# load data

In [3]:
train_df = pd.read_parquet('../../../data/processed/train_btc_4h_native.parquet')
test_df  = pd.read_parquet('../../../data/processed/test_btc_4h_native.parquet')

y_train = train_df.pop('target_direction')
X_train = train_df.drop(columns=['target_4h'], errors='ignore')

y_test = test_df.pop('target_direction')
X_test = test_df.drop(columns=['target_4h'], errors='ignore')

print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   |  y_test:  {y_test.shape}")

X_train: (10579, 33)  |  y_train: (10579,)
X_test:  (2645, 33)   |  y_test:  (2645,)


# mlflow

In [4]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment(f"{ASSET}_{INTERVAL}_{EXPERIMENT}")
mlflow.xgboost.autolog(disable=True)

2026/05/20 11:08:11 INFO mlflow.tracking.fluent: Experiment with name 'BTC_4h_confidence_threshold' does not exist. Creating a new experiment.


# train model with best v4 params

In [5]:
model = xgb.XGBClassifier(
    **BEST_PARAMS,
    tree_method='hist',
    device='cuda',
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.01, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

# Predict all (baseline)

In [6]:
y_pred_all = model.predict(X_test)
y_proba = model.predict_proba(X_test)

base_acc = accuracy_score(y_test, y_pred_all)
base_prec = precision_score(y_test, y_pred_all)
base_rec  = recall_score(y_test, y_pred_all)
base_f1   = f1_score(y_test, y_pred_all)

print(f"BASELINE (predict all {len(y_test):,} rows):")
print(f"  Accuracy:  {base_acc:.4f}")
print(f"  Precision: {base_prec:.4f}")
print(f"  Recall:    {base_rec:.4f}")
print(f"  F1:        {base_f1:.4f}")
print(classification_report(y_test, y_pred_all, target_names=['DOWN', 'UP']))

BASELINE (predict all 2,645 rows):
  Accuracy:  0.5274
  Precision: 0.5245
  Recall:    0.5982
  F1:        0.5589
              precision    recall  f1-score   support

        DOWN       0.53      0.46      0.49      1321
          UP       0.52      0.60      0.56      1324

    accuracy                           0.53      2645
   macro avg       0.53      0.53      0.52      2645
weighted avg       0.53      0.53      0.53      2645



# confidence thresholding — find best threshold

In [7]:
max_proba = np.max(y_proba, axis=1)

thresholds = [0.50, 0.52, 0.54, 0.56, 0.58, 0.60, 0.62, 0.64, 0.66, 0.68, 0.70]

results = []
for thresh in thresholds:
    mask = max_proba >= thresh
    coverage = mask.sum()
    if coverage > 0:
        acc = accuracy_score(y_test[mask], y_pred_all[mask])
        prec = precision_score(y_test[mask], y_pred_all[mask], zero_division=0)
        rec = recall_score(y_test[mask], y_pred_all[mask], zero_division=0)
        f1 = f1_score(y_test[mask], y_pred_all[mask], zero_division=0)
    else:
        acc = prec = rec = f1 = 0.0
    results.append({
        'threshold': thresh,
        'coverage': coverage,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1
    })

results_df = pd.DataFrame(results)
results_df['coverage_pct'] = (results_df['coverage'] / len(y_test) * 100).round(1)
results_df['accuracy'] = results_df['accuracy'].round(4)
results_df['precision'] = results_df['precision'].round(4)
results_df['recall'] = results_df['recall'].round(4)
results_df['f1'] = results_df['f1'].round(4)
results_df['delta_vs_baseline'] = (results_df['accuracy'] - base_acc).round(4)

# Find best threshold (highest accuracy)

In [9]:
best_row = results_df.loc[results_df['accuracy'].idxmax()]

print("=" * 80)
print(f"   CONFIDENCE THRESHOLDING — BTC Native 4h (v4 model)")
print("=" * 80)
print(results_df[['threshold', 'coverage', 'coverage_pct', 'accuracy', 'delta_vs_baseline', 'precision', 'recall', 'f1']].to_string(index=False))
print(f"\n  Best threshold: ≥{best_row['threshold']:.2f} → {best_row['accuracy']:.4f} accuracy (Δ{best_row['delta_vs_baseline']:+.4f}) on {int(best_row['coverage'])} predictions ({best_row['coverage_pct']:.0f}%)")

   CONFIDENCE THRESHOLDING — BTC Native 4h (v4 model)
 threshold  coverage  coverage_pct  accuracy  delta_vs_baseline  precision  recall     f1
      0.50      2645         100.0    0.5274            -0.0000     0.5245  0.5982 0.5589
      0.52      1612          60.9    0.5447             0.0173     0.5440  0.7145 0.6177
      0.54       873          33.0    0.5441             0.0167     0.5444  0.9058 0.6801
      0.56       437          16.5    0.5263            -0.0011     0.5348  0.9449 0.6830
      0.58        15           0.6    0.6667             0.1393     0.6667  1.0000 0.8000
      0.60         1           0.0    0.0000            -0.5274     0.0000  0.0000 0.0000
      0.62         0           0.0    0.0000            -0.5274     0.0000  0.0000 0.0000
      0.64         0           0.0    0.0000            -0.5274     0.0000  0.0000 0.0000
      0.66         0           0.0    0.0000            -0.5274     0.0000  0.0000 0.0000
      0.68         0           0.0    0.0000  